In [12]:
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from snapccess.model import snapshotVAE
from snapccess.train import train_model
from snapccess.util import setup_seed

from sklearn.cluster import KMeans

In [13]:
## read datasets
path = '../in/'
out = '../out/'

ctfile = 'CITEseq_celltypes.csv.gz'
rnafile = 'CITEseq_logRNA.csv.gz'
adtfile = 'CITEseq_logADT.csv.gz'

celltype = pd.read_csv(path+ctfile, index_col=0)

rna = pd.read_csv(path+rnafile, index_col=0).T
rna = rna.reset_index(drop=True)
 
pro = pd.read_csv(path+adtfile, index_col=0).T
pro = pro.reset_index(drop=True)

## get the number of features
nfeatures_rna = rna.shape[1]
nfeatures_pro = pro.shape[1]

In [14]:
## parameters
batch_size = 64
epochs_per_cycle =1
epochs = epochs_per_cycle*100
lr = 0.02
z_dim = 100
hidden_rna2 = 185 
hidden_pro2 = 30 
feature_num = nfeatures_rna + nfeatures_pro 
## standardise each modality of the dataset
rna_sample_scaled=(pd.DataFrame(rna)-pd.DataFrame(rna).mean())/pd.DataFrame(rna).std()
pro_sample_scaled=(pd.DataFrame(pro)-pd.DataFrame(pro).mean())/pd.DataFrame(pro).std()

# combine the standardised modalities to create input data
citeseq = pd.concat([rna_sample_scaled, pro_sample_scaled], axis=1)
train_data=citeseq.to_numpy(dtype=np.float32)

# load data
train_transformed_dataset = train_data
train_dl = DataLoader(train_transformed_dataset, batch_size=batch_size,shuffle=False, num_workers=0,drop_last=False)
test_transformed_dataset = train_data
valid_dl = DataLoader(test_transformed_dataset, batch_size=batch_size, shuffle=False, num_workers=0,drop_last=False)

In [15]:
## run VAE with Snapshot
model = snapshotVAE(num_features=[nfeatures_rna,nfeatures_pro], num_hidden_features=[hidden_rna2,hidden_pro2], z_dim=z_dim).cuda()

In [16]:
for name, layer in model.named_modules():
    print(f"Layer Name: {name}\t Layer Type: {layer.__class__.__name__}")

Layer Name: 	 Layer Type: snapshotVAE
Layer Name: encoder	 Layer Type: Encoder
Layer Name: encoder.encoder_eachmodal	 Layer Type: ModuleList
Layer Name: encoder.encoder_eachmodal.0	 Layer Type: LinBnDrop
Layer Name: encoder.encoder_eachmodal.0.0	 Layer Type: Linear
Layer Name: encoder.encoder_eachmodal.0.1	 Layer Type: ReLU
Layer Name: encoder.encoder_eachmodal.0.2	 Layer Type: BatchNorm1d
Layer Name: encoder.encoder_eachmodal.0.3	 Layer Type: Dropout
Layer Name: encoder.encoder_eachmodal.1	 Layer Type: LinBnDrop
Layer Name: encoder.encoder_eachmodal.1.0	 Layer Type: Linear
Layer Name: encoder.encoder_eachmodal.1.1	 Layer Type: ReLU
Layer Name: encoder.encoder_eachmodal.1.2	 Layer Type: BatchNorm1d
Layer Name: encoder.encoder_eachmodal.1.3	 Layer Type: Dropout
Layer Name: encoder.encoder	 Layer Type: LinBnDrop
Layer Name: encoder.encoder.0	 Layer Type: Linear
Layer Name: encoder.encoder.1	 Layer Type: ReLU
Layer Name: encoder.encoder.2	 Layer Type: BatchNorm1d
Layer Name: encoder.fc_mu

In [17]:
##cuda = True if torch.cuda.is_available() else False
## train the model and generate embeddings, train_dl and valid_dl are the same dataset
model,histroy,embedding = train_model(model, train_dl, valid_dl, lr=lr, epochs=epochs,epochs_per_cycle=epochs_per_cycle, save_path="",snapshot=True,embedding_number=1)

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:29<00:00,  3.41it/s]


In [18]:
print(histroy)

defaultdict(<class 'list'>, {'train': [1.6401093884015445, 1.1768548109652417, 1.0143513523990941, 0.9766112553255959, 0.9711747355261554, 0.9695529879891013, 0.9675363379191418, 0.9654704199661024, 0.9631562711719095, 0.9607567541427307, 0.9584950976677484, 0.9567907745564291, 0.9592616905724483, 0.9557550232641844, 0.9515172829611006, 0.9483971053324739, 0.9464552960858638, 0.9502212349474696, 0.9503336532351595, 0.9457781673114637, 0.9439968744878353, 0.9413300457535742, 0.9369298292395268, 0.9379160486473532, 0.9412602039712513, 0.9429417383638962, 0.9417880070814573, 0.953147049737847, 0.9431742151613651, 0.93592893509385, 0.9332781324191276, 0.9280519099078429, 0.9248413261513783, 0.9268938114042273, 0.9326382184389544, 0.9280816675400161, 0.9256866431512264, 0.9284555317563662, 0.9346036237472949, 0.9309832326344688, 0.9248203033861685, 0.9231261372672376, 0.9222136902575607, 0.9205756648140939, 0.9196672945391994, 0.9206826122127679, 0.9213740102754356, 0.9302327523566948, 0.93

In [19]:
## simple kmeans for one embedding
kmeans = KMeans(n_clusters=4, random_state=0, n_init="auto").fit(embedding[99])
kmeans.labels_

array([1, 1, 1, ..., 1, 3, 1], dtype=int32)

In [21]:
embedding[99]

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
0,1.105237,-0.647851,-1.555419,-2.104080,0.413188,1.916537,2.745037,0.169316,2.473231,-1.395697,...,-4.238571,3.122104,2.111229,-0.441928,-6.815958,-0.694189,-3.958686,-0.443448,-4.563206,-5.141657
1,3.226901,1.560808,-2.396680,-3.953951,-1.475883,-0.531411,3.040147,5.146960,5.713449,-1.055001,...,-2.752539,-2.900792,7.994896,2.051655,-3.816772,-3.822863,-4.184157,-8.880605,4.713256,3.078540
2,4.019595,2.265141,3.336801,0.884509,-1.374815,4.126336,-4.949648,1.196327,1.286448,2.640603,...,-3.002065,-0.867782,0.707336,0.689232,-1.528926,-3.986053,-4.560868,2.416567,2.544457,-0.111952
3,4.603683,2.544142,3.348171,-0.262048,-3.264229,-0.713970,4.254947,-1.904782,3.691066,-0.430939,...,-0.013048,0.157778,-0.349950,10.641189,1.757666,-2.221127,-4.595534,-1.599841,-2.147623,-4.314299
4,3.049155,-1.933268,-1.271604,-0.288781,-3.447044,-3.551167,1.134849,0.469576,4.187469,1.657526,...,-1.120061,3.559349,-4.165165,2.331771,8.360767,-0.214552,-1.459883,0.577603,-1.553944,-1.030404
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1118,7.791610,10.818482,-0.020893,3.182485,-3.805276,-3.643076,-5.776857,1.455733,-1.134873,-5.586783,...,2.386635,-3.687740,4.483288,-3.918340,3.431640,-3.377550,-1.505103,2.488121,6.012195,0.686742
1119,8.886260,13.456982,-2.636696,13.022223,-11.971951,-4.205584,2.862217,-1.040985,9.227975,14.556929,...,-0.118728,-13.598432,11.589813,13.307286,-17.625631,15.570017,-0.646099,9.433042,-6.298013,5.584305
1120,3.013348,-1.636849,4.589317,0.986699,3.447007,2.294765,7.261733,5.073704,1.557421,-6.281219,...,-6.997522,-0.209713,0.405100,-8.110838,-3.013489,3.766306,-1.241735,-7.777497,1.355993,-4.271767
1121,5.579428,10.100705,-10.389549,6.771537,-4.280117,10.058018,12.030406,19.358242,-3.834759,0.931044,...,6.923150,2.733951,-12.290783,12.015494,-7.431779,0.958231,-2.187771,-3.758507,-5.182451,-0.602415


In [20]:
labels_df = pd.DataFrame({'Cluster Labels': kmeans.labels_})

# Save the labels to a CSV file
labels_df.to_csv(out + 'clustering.csv', index=False)

In [11]:
## save all embeddings
for ind,eb in enumerate(embedding):
    eb.to_csv(out+'CITEseq'+'_embedding_{}.csv.gz'.format(ind),
        index=False,compression="gzip")